In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path
import json
import numpy as np
import random

load_dotenv()

# 로컬
# ROOT = Path(os.environ["DATA_ROOT"])
# HF_HOME = ROOT / ".hf_cache"
# os.environ["HF_HOME"] = str(HF_HOME)

# 클라우드
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HUGGINGFACEHUB_API_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = "patent_disc"
os.environ["HF_HOME"] = ".hf_cache"

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import load_dataset
from sklearn.metrics import f1_score

In [ ]:
# Config
config = {
    'num_labels': 188,
    'seed': 42,
    'learning_rate': 3e-5,
    'batch_size': 8,
    'epochs': 12,
    'early_stop': 5,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'model_name': 'monologg/kobert',
    "repo": "ingyoun/kobert-patent-baseline",
    "run_name": "kobert-patent-baseline"
}

In [ ]:
random.seed(config['seed'])
np.random.seed(config['seed'])
torch.manual_seed(config['seed'])
torch.cuda.manual_seed_all(config['seed'])

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## 데이터셋

In [ ]:
dataset = load_dataset("ingyoun/patent-clean-text-kobert-tokenized")
dataset

README.md:   0%|          | 0.00/600 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/543M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/530M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/60.0M [00:00<?, ?B/s]

data/val-00000-of-00001.parquet:   0%|          | 0.00/59.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/201895 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11271 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/11162 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['document_id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 201895
    })
    test: Dataset({
        features: ['document_id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11271
    })
    val: Dataset({
        features: ['document_id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11162
    })
})

## 토크나이저

In [ ]:
REV = "38279184ba645e8c94d709fbe92eb5bcb47312c1"
tokenizer = AutoTokenizer.from_pretrained(config["model_name"], trust_remote_code=True, revision=REV)

config.json:   0%|          | 0.00/426 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/263 [00:00<?, ?B/s]

tokenization_kobert.py:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

tokenizer_78b3253a26.model:   0%|          | 0.00/371k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/77.8k [00:00<?, ?B/s]

In [ ]:
_orig_save_vocab = tokenizer.save_vocabulary
tokenizer.save_vocabulary = lambda save_directory, filename_prefix=None: _orig_save_vocab(save_directory)

## 모델

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
        pretrained_model_name_or_path=config["model_name"],
        num_labels=config["num_labels"],
        problem_type="multi_label_classification",
        classifier_dropout=0.5
    )

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: monologg/kobert
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha: float = 1.0, gamma: int = 2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        pt = torch.exp(-bce)
        return (self.alpha * (1 - pt) ** self.gamma * bce).mean()

In [ ]:
class FocalTrainer(Trainer):
    def __init__(self, *a, **k):
        super().__init__(*a, **k)
        self.focal = FocalLoss(0.25, 2.0)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs["labels"]
        outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        loss = self.focal(outputs.logits, labels.float())
        return (loss, outputs) if return_outputs else loss

In [ ]:
class MultiLabelCollator:
    def __init__(self, tokenizer):
        self.tok = tokenizer

    def __call__(self, feats):
        labels = torch.tensor([f["labels"] for f in feats], dtype=torch.float)
        keys = ("input_ids", "attention_mask")
        enc = [{k: f[k] for k in keys if k in f} for f in feats]
        batch = self.tok.pad(enc, padding=True, return_tensors="pt")
        batch["labels"] = labels
        return batch

In [ ]:
def evaluate_topk(logits, multihot):                        # logits/multihot: [N,188]
    pred_top1 = logits.argmax(axis=1)                       # top-1 예측
    gold_top1 = multihot.argmax(axis=1)                     # 정답 단일화(원본 LabelBinarizer.inverse_transform과 등가)
    out = {
        "weighted_f1": f1_score(gold_top1, pred_top1, average="weighted"),  # baseline headline과 동일
        "micro_f1":    f1_score(gold_top1, pred_top1, average="micro"),
        "macro_f1":    f1_score(gold_top1, pred_top1, average="macro"),
    }
    order = np.argsort(-logits, axis=1)                     # P@k (멀티레이블 참고 지표)
    for k in (1, 3, 5):
        topk = order[:, :k]
        hit = np.take_along_axis(multihot, topk, axis=1).sum(1)
        out[f"p@{k}"] = float((hit / np.clip(multihot.sum(1), 1, k)).mean())
    return out


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return evaluate_topk(np.asarray(logits), np.asarray(labels))

In [ ]:
training_args = TrainingArguments(
    output_dir='/content/results',
    seed=config["seed"],
    learning_rate=config["learning_rate"],
    weight_decay=config["weight_decay"],
    lr_scheduler_type="linear",
    warmup_ratio=config["warmup_ratio"],
    per_device_train_batch_size=config["batch_size"],
    per_device_eval_batch_size=config["batch_size"],
    num_train_epochs=config["epochs"],
    fp16=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    logging_dir='/content/logs',
    logging_steps=50,
    metric_for_best_model="weighted_f1",
    load_best_model_at_end=True,
    push_to_hub=True,
    hub_model_id=config["repo"],
    hub_strategy="checkpoint",
    report_to="wandb",
    run_name=config["run_name"]
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer = FocalTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    data_collator=MultiLabelCollator(tokenizer),
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=config["early_stop"])]
)

In [ ]:
# trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: paraise (paraise-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Weighted F1,Micro F1,Macro F1,P@1,P@3,P@5
1,0.000956,0.000860,0.630710,0.648361,0.594733,0.715463,0.859762,0.908974
2,0.000589,0.000601,0.737137,0.741892,0.707138,0.818312,0.926223,0.957979
3,0.000573,0.000519,0.766499,0.768590,0.737313,0.847608,0.941632,0.969560


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download("ingyoun/kobert-patent-baseline",
                  local_dir="/content/results",
                  allow_patterns="last-checkpoint/*")

trainer.train(resume_from_checkpoint="/content/results/last-checkpoint")

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: paraise (paraise-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Weighted F1,Micro F1,Macro F1,P@1,P@3,P@5
4,0.000501,0.000471,0.785137,0.786239,0.757836,0.865974,0.952338,0.975633
5,0.000347,0.000463,0.788877,0.789016,0.762410,0.872514,0.955862,0.976560
6,0.000378,0.000469,0.788132,0.789554,0.762646,0.872066,0.952816,0.973599
7,0.000311,0.000439,0.804568,0.805322,0.779329,0.884877,0.962552,0.978999
8,0.000258,0.000441,0.807764,0.808636,0.784567,0.888998,0.963507,0.980480
9,0.000184,0.000455,0.808001,0.808547,0.783642,0.892224,0.963403,0.980407
10,0.000165,0.000480,0.810567,0.811682,0.787326,0.892940,0.963089,0.979689


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Weighted F1,Micro F1,Macro F1,P@1,P@3,P@5
4,0.000501,0.000471,0.785137,0.786239,0.757836,0.865974,0.952338,0.975633
5,0.000347,0.000463,0.788877,0.789016,0.762410,0.872514,0.955862,0.976560
6,0.000378,0.000469,0.788132,0.789554,0.762646,0.872066,0.952816,0.973599
7,0.000311,0.000439,0.804568,0.805322,0.779329,0.884877,0.962552,0.978999
8,0.000258,0.000441,0.807764,0.808636,0.784567,0.888998,0.963507,0.980480
9,0.000184,0.000455,0.808001,0.808547,0.783642,0.892224,0.963403,0.980407
10,0.000165,0.000480,0.810567,0.811682,0.787326,0.892940,0.963089,0.979689
11,0.000138,0.000496,0.813389,0.813922,0.788189,0.894463,0.964179,0.980275
12,0.000082,0.000517,0.810523,0.811145,0.785696,0.894015,0.963955,0.978773


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=302844, training_loss=0.00019610258395133385, metrics={'train_runtime': 27037.1696, 'train_samples_per_second': 89.608, 'train_steps_per_second': 11.201, 'total_flos': 6.38512715889153e+17, 'train_loss': 0.00019610258395133385, 'epoch': 12.0})

## 평가

In [ ]:
test_metrics = trainer.evaluate(dataset["test"], metric_key_prefix="test")
for k, v in test_metrics.items():
    print(f"{k}: {v}")

[transformers] early stopping required metric_for_best_model, but did not find eval_weighted_f1 so early stopping is disabled


Training Loss,Validation Loss,Epoch,Weighted F1,Micro F1,Macro F1,P@1,P@3,P@5
0.000082,0.000505,12,0.814782,0.814746,0.787030,0.893710,0.962337,0.979112


test_loss: 0.0005048275925219059
test_weighted_f1: 0.8147823819212783
test_micro_f1: 0.8147458078253926
test_macro_f1: 0.7870298971957447
test_p@1: 0.8937095403671265
test_p@3: 0.9623369574546814
test_p@5: 0.9791115522384644


In [ ]:
OUT = "/content/out/kobert-baseline"
os.makedirs(OUT, exist_ok=True)
trainer.save_model(OUT)
tokenizer.save_pretrained(OUT)

with open(f"{OUT}/test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, ensure_ascii=False, indent=2)

print("saved", OUT)

In [ ]:
trainer.push_to_hub(commit_message="Upload final model")
tokenizer.push_to_hub(commit_message="Upload final tokenizer")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

No files have been modified since last commit. Skipping to prevent empty commit.


README.md:   0%|          | 0.00/3.05k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...okenizer_78b3253a26.model: 100%|##########|  371kB /  371kB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/ingyoun/kobert-patent-baseline/commit/64703c957352d3395b825f58b0395fe6d9d531f1', commit_message='Upload tokenizer', commit_description='', oid='64703c957352d3395b825f58b0395fe6d9d531f1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ingyoun/kobert-patent-baseline', endpoint='https://huggingface.co', repo_type='model', repo_id='ingyoun/kobert-patent-baseline'), pr_revision=None, pr_num=None)